In [2]:
from transformers import (
    AutoTokenizer, BertForMaskedLM, BertConfig, TrainingArguments, Trainer, DataCollatorForWholeWordMask,
    GenerationConfig
)
from datasets import load_dataset, Dataset, DatasetDict
import torch
import pandas as pd
import numpy as np
from typing import Any, Union, List, Tuple, Dict
from transformers.data.data_collator import DataCollatorMixin, InputDataClass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import nltk
from nltk.corpus import stopwords

In [3]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

In [4]:
def select_random_words(x, num_words=5):
    x = x.split(' ')
    try:
        x.remove('-')
        x.remove('')
    except:
        pass
    filtered_words = []
    for word in x:
        if not x in stopwords.words('english'):
            filtered_words.append(word)
    return np.random.choice(np.array(filtered_words), num_words, replace=False).tolist()

def chunk_text(x):
    text_splitter = RecursiveCharacterTextSplitter(
        # Set a really small chunk size, just to show.
        chunk_size=15,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False,
    )
    res = list(set([chunk.page_content for chunk in text_splitter.create_documents(eval(x))]))
    filtered_words = []
    for word in res:
        if not x in stopwords.words('english'):
            filtered_words.append(word)
    try:
        filtered_words.remove('-')
    except:
        pass
    return filtered_words

In [5]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")

In [6]:
df['assetlongdescription_entity_llms'] = df['assetlongdescription_entity_llms'].apply(chunk_text)
is_empty = ~df['assetlongdescription_entity_llms'].astype(bool)
random_words = df['answers.text'].apply(select_random_words)
df.loc[is_empty, 'assetlongdescription_entity_llms'] = random_words

In [7]:
include_failure_locations = True

In [8]:
if include_failure_locations:
    df['failurelocation_original'] = df['failurelocation_original'].apply(lambda x: list(eval(x)))
    df['assetlongdescription_entity_llms'] = df['failurelocation_original'] + df['assetlongdescription_entity_llms']

In [9]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [64]:
chkpt_path = '/dccstor/chrisconst2/AutoQA/fmea_recommender/entity_masking_mlm/checkpoint-14000'
# chkpt_path = 'google-bert/bert-base-uncased'

In [65]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
config = BertConfig.from_pretrained(chkpt_path, output_hidden_states=False)
model = BertForMaskedLM.from_pretrained(chkpt_path, config=config)

In [66]:
generation_config = GenerationConfig(
    max_new_tokens=512,
    min_new_tokens=2,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)

In [67]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        item = {
                'input_ids': self.df.iloc[idx]['answers.text'],
                'failure_locations': self.df.iloc[idx]['failurelocation_original'],
                'labels': self.df.iloc[idx].assetlongdescription_entity_llms
               }
        return item

In [68]:
ds_train = CustomDataset(df_train)
ds_val = CustomDataset(df_val)
ds_test = CustomDataset(df_test)

In [69]:
def collate_fn(data, mask_prob=0.2):
    batch = {'original_text': [], 'masked_text': [], 'failure_locations': []}
    for i in range(len(data)):
        text = data[i]['input_ids']
        failure_locations = 'Failure locations: ' + str(data[i]['failure_locations'])
        batch['original_text'].append(text)
        batch['failure_locations'].append(failure_locations)
    if include_failure_locations:
        original_tokenized = tokenizer(batch['original_text'], batch['failure_locations'], 
                                       padding="max_length", max_length=512, truncation=True, 
                                       return_tensors='pt')
    else:
        original_tokenized = tokenizer(batch['original_text'], padding="max_length", 
                                       max_length=512, truncation=True, return_tensors='pt')
    attention_masks = original_tokenized.attention_mask
    mask_tokenized = original_tokenized['input_ids'].detach().clone()
    mask_token_id = torch.tensor(tokenizer.mask_token_id, dtype=torch.long)
    all_labels = []
    for idx, original_tokens in enumerate(original_tokenized['input_ids']):
        entities = np.array(data[idx]['labels'])
        original_text = text
        sel_idxs = np.random.binomial(1, mask_prob, len(entities)).astype(bool)
        masked_entities = entities[sel_idxs]
        mask_idxs = torch.zeros(len(original_tokens)).bool()
        for entity in masked_entities:
            tokenized_entity = tokenizer(entity, return_tensors='pt')['input_ids'][0, 1:-1]
            for i in range(len(original_tokens) - len(tokenized_entity) + 1):
                if torch.equal(original_tokens[i], mask_token_id):
                    break
                original_token_slice = original_tokens[i:i+len(tokenized_entity)]
                if torch.equal(original_token_slice, tokenized_entity):
                    mask_idxs[i:i+len(tokenized_entity)] = True
        labels = torch.where(mask_idxs, original_tokens, -100)
        mask_tokenized[idx][mask_idxs] = mask_token_id
        all_labels.append(labels)
    # 1 mask prediction at a time
    results = {
        'input_ids': mask_tokenized,
        'labels': torch.stack(all_labels),
        'token_type_ids': original_tokenized['token_type_ids'],
        'attention_mask': attention_masks
    }
    return results

In [70]:
def compare_preds(item, outputs, limit=1):
    preds = torch.argmax(outputs, axis=2)
    masked_str = tokenizer.batch_decode(item['input_ids'])
    for i in range(min(len(preds), limit)):
        mask_idxs = item['input_ids'][i] == tokenizer.mask_token_id
        pred_tokens = preds[i][mask_idxs]
        gt_tokens = item['labels'][i][mask_idxs]
        predicted_str = tokenizer.decode(pred_tokens).replace('[PAD]', '')
        gt_str = tokenizer.decode(gt_tokens).replace('[PAD]', '')
        print("Input"+"*"+"*"*100)
        print(masked_str[i].replace('[PAD]', ''))
        print("*"*100)
        predicted_str = predicted_str.replace('[PAD]', '')
        print(f'Predicted:{bcolors.OKGREEN}{predicted_str}{bcolors.ENDC}')
        print(f'    Label:{bcolors.OKCYAN}{gt_str}{bcolors.ENDC}')

In [71]:
dataloader = DataLoader(ds_test, batch_size=4, collate_fn=collate_fn)

In [72]:
item = next(iter(dataloader))

In [73]:
item = next(iter(dataloader))
# outputs = model.generate(**item, generation_config=generation_config)
outputs = model(**item)
compare_preds(item, outputs.logits, limit=4)

Input*****************************************************************************************************
[CLS] the equipment accumulator - hydraulic - bladder type, is categorized as fixed asset and has the following boundary : a hydraulic accumulator - bladder type in this database is comprised of : - tank - bladder - oil check valve - [MASK] [MASK] [MASK] [MASK] valve [SEP] failure locations : ['oil valve ','bladder ','gas fill valve'] [SEP]                                                                                                                                                                                                                                                                                                                                                                                                                                                       
****************************************************************************************************
Predicted:

In [24]:
training_args = TrainingArguments(
    output_dir="entity_masking_mlm",
    evaluation_strategy="epoch",
    num_train_epochs=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    remove_unused_columns=False
    
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate_fn
)


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [16]:
data = [ds_val[i] for i in range(8)]
collated_data = collate_fn(data)
outputs = trainer.predict(data)
preds, hidden_states = outputs.predictions
compare_preds(collated_data, torch.Tensor(preds), limit=32)
# outputs = trainer.predict(questions)

NameError: name 'trainer' is not defined

In [17]:
model = trainer.model
tokenized = tokenizer('battery - charger', return_tensors='pt')
output = model(**tokenized)

In [24]:
hidden_states = output.hidden_states

In [27]:
# last layer embeddings: (batch_size, num_tokens, embedding_dim)
hidden_states[-1].shape

torch.Size([1, 6, 768])

In [ ]:
# todo: get the embeddings and do analysis